# MedGemma reader on a free Kaggle GPU

Runs `python/scripts/11_ai_predict.py` from [health-transformation-lab-nlp](https://github.com/jcbowyer/health-transformation-lab-nlp)
with MedGemma on Kaggle's free GPU, for when Databricks Free Edition has no GPU to give. Same code, same
prompts and the same 240 **synthetic** review notes as the Databricks job; nothing here touches real
patient data.

**Before you start (once):**

1. A Kaggle account with a **verified phone number** (Kaggle requires it for GPUs and internet).
2. No Hugging Face token: MedGemma comes from Kaggle Models,
   [keras/medgemma](https://www.kaggle.com/models/keras/medgemma) (Keras preset, used under the Health AI
   Developer Foundations terms), and cell 1 converts it to the layout `transformers` reads. The GitHub workflow
   attaches it; by hand, **Add Input → Models → keras/medgemma**, variation `medgemma_1.5_instruct_4b`.
3. In the right-hand panel: **Accelerator → GPU T4 x2** (two GPUs, 30 GB: room for MedGemma in float32;
   the T4 has no native bfloat16, and Gemma models can overflow in float16) and **Internet → On**.

Then run the cells in order. Do the 10-note test first; for the full run use **Save Version → Save & Run All
(Commit)** so it carries on after you close the browser, and download the results from the **Output** tab.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Settings

In [ ]:
RUNG = "medgemma-1.5-4b-it"   # or medgemma-4b-it (models.yaml)
SHOTS = 0                     # few-shot examples per status (0 = zero-shot, as in the README results)
TEST_NOTES = 10               # quick check before the full 240
BRANCH = "main"
COMMIT = ""                   # a commit of BRANCH to run; empty: its latest (the workflow pins it)
FULL_RUN = False              # True: also read all 240 notes (cell 2); set by the GitHub workflow


## Get the code and MedGemma from Kaggle Models

Converts the Keras preset to a `transformers` folder (bfloat16, as Google publishes it; about 10 minutes).

In [ ]:
import os

REPO = "/kaggle/temp/health-transformation-lab-nlp"      # outside /kaggle/working, so neither is saved as output
WEIGHTS = "/kaggle/temp/medgemma-hf"
os.makedirs("/kaggle/temp", exist_ok=True)
!git clone --quiet --depth 1 -b {BRANCH} https://github.com/jcbowyer/health-transformation-lab-nlp {REPO}
if COMMIT:
    !git -C {REPO} fetch --quiet --depth 1 origin {COMMIT} && git -C {REPO} checkout --quiet FETCH_HEAD
!git -C {REPO} log -1 --date=short --format="code: {BRANCH} @ %h (%cd) %s"
!pip install --quiet -e "{REPO}/python[llm]" "transformers>=4.52" "keras-hub==0.32.0"
os.environ["DENTAL_TELEMETRY"] = "off"
!python {REPO}/kaggle/keras_to_hf.py --rung {RUNG} --out {WEIGHTS}
assert os.path.exists(f"{WEIGHTS}/model.safetensors"), "conversion failed: see the output above"
print("ready")

## 1. Test on a few notes

Loads the model (a few minutes the first time) and reads `TEST_NOTES` notes. The log says which dtype it
picked: float32 on T4 x2.

In [ ]:
!cd {REPO}/python && python scripts/11_ai_predict.py --rung {RUNG} --backend hf --weights {WEIGHTS} --shots {SHOTS} --limit {TEST_NOTES}

## 2. Full run: all 240 review notes

Several minutes per 10 notes on T4 x2, so a few hours in all: run it with **Save & Run All (Commit)**. Set `FULL_RUN = True` in the settings first.


In [ ]:
if FULL_RUN:
    !cd {REPO}/python && python scripts/11_ai_predict.py --rung {RUNG} --backend hf --weights {WEIGHTS} --shots {SHOTS}
else:
    print("FULL_RUN is False: skipping the 240-note run (set it to True above, or run the workflow with limit 0)")


## 3. Keep the results

Copies the prediction files to `/kaggle/working`, which is what the **Output** tab offers for download.

In [ ]:
import glob, shutil
files = glob.glob(f"{REPO}/data/datasets/*_{RUNG}*.csv")
for f in files:
    if not os.path.basename(f).startswith("simulated_"):
        shutil.copy(f, "/kaggle/working/")
        print("saved", os.path.basename(f))

## Back in the repo

1. Put the downloaded `ai_predictions_<rung>.csv` and `llm_details_<rung>.csv` in `data/datasets/`.
2. `cd python && python scripts/28_ladder_vs_truth.py && python scripts/35_readme_results.py`
3. Commit: the README and `docs/results.md` then show MedGemma's real (not simulated) scores.